# 第一课：张量与自动求导（可视化版）

**学习目标**：理解 TF 的数据结构（张量）和核心引擎（自动求导），全部用图表展示。

> 对应 PyTorch：`torch.Tensor` + `torch.autograd`

In [9]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import os; os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
print('TF版本:', tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

## 1. 张量的维度

0维=标量，1维=向量，2维=矩阵，3维+ =高维张量（图片批次）。

In [2]:
scalar = tf.constant(3.14)
vector = tf.constant([1, 2, 3, 4])
matrix = tf.constant([[1,2,3],[4,5,6],[7,8,9]])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(['scalar'], [scalar.numpy()], color='steelblue')
axes[0].set_title('0D 标量: %.2f' % scalar.numpy())
axes[0].set_ylim(0, 4)
axes[1].bar(range(4), vector.numpy(), color='coral')
axes[1].set_title('1D 向量 shape=%s' % str(vector.shape))
axes[1].set_xticks(range(4))
im = axes[2].imshow(matrix.numpy(), cmap='YlOrRd')
axes[2].set_title('2D 矩阵 shape=%s' % str(matrix.shape))
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, matrix[i,j].numpy(), ha='center', va='center')
plt.colorbar(im, ax=axes[2])
plt.tight_layout(); plt.show()

NameError: name 'tf' is not defined

### 1.1 模拟一批图片（4D张量）

图片数据格式：`(batch, height, width, channel)`

In [3]:
batch = tf.random.normal((8, 64, 64, 3))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = batch[i].numpy()
    img = (img - img.min()) / (img.max() - img.min())
    ax.imshow(img); ax.set_title('图片 %d' % (i+1)); ax.axis('off')
plt.suptitle('4D张量 shape=%s (batch, H, W, C)' % str(batch.shape), fontsize=14)
plt.tight_layout(); plt.show()

NameError: name 'tf' is not defined

## 2. 张量运算可视化

In [4]:
a = tf.constant([[1,2],[3,4]], dtype=tf.float32)
b = tf.constant([[5,6],[7,8]], dtype=tf.float32)
all_mats = [('a',a), ('b',b), ('a+b',a+b), ('a*b',a*b), ('a@b',a@b)]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (title, mat) in zip(axes, all_mats):
    im = ax.imshow(mat.numpy(), cmap='coolwarm', vmin=-5, vmax=60)
    ax.set_title(title)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, '%.0f'%mat[i,j].numpy(), ha='center', va='center', fontsize=12)
plt.colorbar(im, ax=axes, shrink=0.8)
plt.tight_layout(); plt.show()

NameError: name 'tf' is not defined

## 3. 自动求导 GradientTape

在 `tape` 上下文里做运算，TF 自动记录，结束后 `tape.gradient()` 一键求导。

> 对应 PyTorch：`loss.backward()`

### 3.1 f(x) = x² 的导数（函数曲线 + 切线）

In [5]:
x = tf.Variable(3.0)
with tf.GradientTape() as tape:
    y = x ** 2
dy_dx = tape.gradient(y, x)

x_range = np.linspace(-5, 5, 100)
x0, y0, slope = x.numpy(), y.numpy(), dy_dx.numpy()
tangent = slope * (x_range - x0) + y0

plt.figure(figsize=(8, 5))
plt.plot(x_range, x_range**2, 'b-', lw=2, label='f(x)=x²')
plt.plot(x_range, tangent, 'r--', lw=2, label='切线(斜率=%.1f)'%slope)
plt.scatter([x0],[y0], color='red', s=100, zorder=5, label='点(%.1f,%.1f)'%(x0,y0))
plt.title('f(x)=x² 在 x=%.1f 的导数=%.1f (数学解:2x=%.1f)'%(x0,slope,2*x0))
plt.legend(); plt.grid(alpha=0.3); plt.xlabel('x'); plt.ylabel('y')
plt.axhline(0,color='k',lw=0.5); plt.axvline(0,color='k',lw=0.5)
plt.show()

NameError: name 'tf' is not defined

### 3.2 梯度下降全过程可视化

In [6]:
x = tf.Variable(4.0)
lr = 0.1
traj = []
for step in range(20):
    with tf.GradientTape() as tape:
        y = x ** 2
    grad = tape.gradient(y, x)
    traj.append((x.numpy(), y.numpy()))
    x.assign_sub(lr * grad)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
xr = np.linspace(-4.5, 4.5, 100)
axes[0].plot(xr, xr**2, 'b-', alpha=0.5, label='f(x)=x²')
axes[0].plot([t[0] for t in traj], [t[1] for t in traj], 'ro-', ms=5, label='下降轨迹')
axes[0].scatter([0],[0], color='green', s=200, marker='*', zorder=5, label='最小值x=0')
axes[0].set_title('函数曲线上的梯度下降轨迹')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x)')
axes[1].plot(range(len(traj)), [t[0] for t in traj], 'b-o', ms=4, label='x值')
axes[1].plot(range(len(traj)), [t[1] for t in traj], 'r-s', ms=4, label='损失')
axes[1].axhline(0, color='g', ls='--', alpha=0.5)
axes[1].set_title('x和损失随步数变化')
axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_xlabel('步数')
plt.tight_layout(); plt.show()
print('最终 x=%.4f, 损失=%.6f' % (x.numpy(), x.numpy()**2))

NameError: name 'tf' is not defined

### 3.3 多变量求导：y = wx + b

In [7]:
w = tf.Variable(2.0); b = tf.Variable(1.0); xv = 3.0
with tf.GradientTape() as tape:
    yp = w * xv + b
grads = tape.gradient(yp, [w, b])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
wr = np.linspace(0, 4, 50)
axes[0].plot(wr, wr*xv+b.numpy(), 'b-', lw=2)
axes[0].scatter([w.numpy()], [yp.numpy()], color='red', s=100, zorder=5)
axes[0].annotate('dy/dw=%.1f\n(=x=%.1f)'%(grads[0].numpy(),xv), xy=(w.numpy(),yp.numpy()),
                 xytext=(2.5,5), arrowprops=dict(arrowstyle='->'))
axes[0].set_title('w对输出的影响'); axes[0].set_xlabel('w'); axes[0].set_ylabel('y=wx+b')
axes[0].grid(alpha=0.3)
br = np.linspace(-2, 4, 50)
axes[1].plot(br, w.numpy()*xv+br, 'g-', lw=2)
axes[1].scatter([b.numpy()], [yp.numpy()], color='red', s=100, zorder=5)
axes[1].annotate('dy/db=%.1f\n(恒等于1)'%grads[1].numpy(), xy=(b.numpy(),yp.numpy()),
                 xytext=(1.5,5), arrowprops=dict(arrowstyle='->'))
axes[1].set_title('b对输出的影响'); axes[1].set_xlabel('b'); axes[1].set_ylabel('y=wx+b')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

NameError: name 'tf' is not defined

## 4. Variable（可变参数）vs constant（不可变）

In [8]:
v = tf.Variable([1.0,2.0,3.0])
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(['v1','v2','v3'], v.numpy(), color='steelblue')
axes[0].set_title('Variable初始值'); axes[0].set_ylim(0,8)
v.assign(v * 2)
axes[1].bar(['v1','v2','v3'], v.numpy(), color='coral')
axes[1].set_title('v.assign(v*2)后'); axes[1].set_ylim(0,8)
c = tf.constant([1.0,2.0,3.0])
axes[2].bar(['c1','c2','c3'], (c+1).numpy(), color='green')
axes[2].set_title('constant c+1返回新张量\n(原c不变)'); axes[2].set_ylim(0,8)
plt.tight_layout(); plt.show()

NameError: name 'tf' is not defined

## 第一课小结

| 概念 | 作用 | 对应 PyTorch |
|---|---|---|
| `tf.constant` | 不可变张量 | `torch.tensor` |
| `tf.Variable` | 可变参数 | `nn.Parameter` |
| `tf.GradientTape` | 自动求导 | `loss.backward()` |
| `tape.gradient(loss,vars)` | 计算梯度 | `param.grad` |
| `optimizer.apply_gradients` | 更新参数 | `optimizer.step()` |

**下一课**：线性回归，可视化拟合全过程。